In [23]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub

/kaggle/input/competitions/csiro-biomass/sample_submission.csv
/kaggle/input/competitions/csiro-biomass/train.csv
/kaggle/input/competitions/csiro-biomass/test.csv
/kaggle/input/competitions/csiro-biomass/test/ID1001187975.jpg
/kaggle/input/competitions/csiro-biomass/train/ID2099464826.jpg
/kaggle/input/competitions/csiro-biomass/train/ID2037861084.jpg
/kaggle/input/competitions/csiro-biomass/train/ID1211362607.jpg
/kaggle/input/competitions/csiro-biomass/train/ID1853508321.jpg
/kaggle/input/competitions/csiro-biomass/train/ID193102215.jpg
/kaggle/input/competitions/csiro-biomass/train/ID698608346.jpg
/kaggle/input/competitions/csiro-biomass/train/ID1859251563.jpg
/kaggle/input/competitions/csiro-biomass/train/ID1880764911.jpg
/kaggle/input/competitions/csiro-biomass/train/ID853954911.jpg
/kaggle/input/competitions/csiro-biomass/train/ID1403107574.jpg
/kaggle/input/competitions/csiro-biomass/train/ID1781353117.jpg
/kaggle/input/competitions/csiro-biomass/train/ID384648061.jpg
/kaggle/i

In [24]:

import kagglehub

# Download latest version
path = kagglehub.competition_download('csiro-biomass')

print("Path to competition files:", path)

Path to competition files: /kaggle/input/competitions/csiro-biomass


Task: Predict pasture biomass from top‑view RGB images of fields.

In [25]:
train=pd.read_csv('/kaggle/input/competitions/csiro-biomass/train.csv')
test=pd.read_csv('/kaggle/input/competitions/csiro-biomass/test.csv')

In [26]:
print(train.shape)
print(test.shape)
print(train.columns)
print(test.columns)

(1785, 9)
(5, 3)
Index(['sample_id', 'image_path', 'Sampling_Date', 'State', 'Species',
       'Pre_GSHH_NDVI', 'Height_Ave_cm', 'target_name', 'target'],
      dtype='object')
Index(['sample_id', 'image_path', 'target_name'], dtype='object')


In [37]:

BASE_DIR = '/kaggle/input/competitions/csiro-biomass'

def load_image(path, size=(128, 128)):
    full_path = os.path.join(BASE_DIR, path)
    img = tf.keras.utils.load_img(full_path, target_size=size)
    img = tf.keras.utils.img_to_array(img)
    img = img / 255.0
    return img


In [38]:
X = np.array([load_image(p) for p in train['image_path']])
y = train['target'].values

In [45]:
import tensorflow as tf
from tensorflow.keras import layers,models
model=models.Sequential([
    layers.Input(shape=(128, 128, 3)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1)
    
])

In [56]:
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae","accuracy"]
)

In [57]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(X,y,test_size=0.2)
model.fit(x_train,y_train)
history = model.fit(
    x_train, y_train,
    validation_data=(x_test, y_test),
    epochs=5,
    batch_size=32
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 24s 494ms/step - accuracy: 7.0028e-04 - loss: 689.2589 - mae: 19.2447
Epoch 1/5
45/45 ━━━━━━━━━━━━━━━━━━━━ 24s 537ms/step - accuracy: 7.0028e-04 - loss: 647.3372 - mae: 18.8096 - val_accuracy: 0.0000e+00 - val_loss: 565.6390 - val_mae: 18.0172
Epoch 2/5
45/45 ━━━━━━━━━━━━━━━━━━━━ 23s 521ms/step - accuracy: 7.0028e-04 - loss: 622.8145 - mae: 18.4120 - val_accuracy: 0.0000e+00 - val_loss: 621.0421 - val_mae: 20.3474
Epoch 3/5
45/45 ━━━━━━━━━━━━━━━━━━━━ 41s 521ms/step - accuracy: 7.0028e-04 - loss: 615.1931 - mae: 18.5546 - val_accuracy: 0.0000e+00 - val_loss: 622.9775 - val_mae: 17.7842
Epoch 4/5
45/45 ━━━━━━━━━━━━━━━━━━━━ 24s 523ms/step - accuracy: 7.0028e-04 - loss: 605.4197 - mae: 18.4002 - val_accuracy: 0.0000e+00 - val_loss: 590.5732 - val_mae: 19.1699
Epoch 5/5
45/45 ━━━━━━━━━━━━━━━━━━━━ 24s 524ms/step - accuracy: 7.0028e-04 - loss: 584.2382 - mae: 18.2089 - val_accuracy: 0.0000e+00 - val_loss: 560.9188 - val_mae: 18.0772


In [62]:
preds=model.predict(x_test)
mae = np.mean(np.abs(preds - y_test) <= 5)
print(mae)

12/12 ━━━━━━━━━━━━━━━━━━━━ 2s 135ms/step
0.1464978148121994


In [64]:
import mlflow
import mlflow.tensorflow
mlflow.set_experiment("csiro_biomass_baseline")

2026/07/25 17:59:55 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/25 17:59:55 INFO mlflow.store.db.utils: Updating database tables
2026/07/25 17:59:59 INFO mlflow.tracking.fluent: Experiment with name 'csiro_biomass_baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='/kaggle/working/mlruns/1', creation_time=1785002399088, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1785002399088, lifecycle_stage='active', name='csiro_biomass_baseline', tags={}, trace_location=None, workspace='default'>

In [ ]:
with mlflow.start_run():
    mlflow.log_param("image_size", 128)
    mlflow.log_param("epochs", 5)
    mlflow.log_param("batch_size", 32)

    history = model.fit(
        x_train, y_train,
        validation_data=(x_test, y_test),
        epochs=5,
        batch_size=32
    )

    mlflow.log_metric("val_mae", history.history["val_mae"][-1])
    mlflow.log_metric("val_loss", history.history["val_loss"][-1])

    mlflow.tensorflow.log_model(model, "model")

Epoch 1/5
45/45 ━━━━━━━━━━━━━━━━━━━━ 24s 531ms/step - accuracy: 7.0028e-04 - loss: 558.6599 - mae: 17.6629 - val_accuracy: 0.0000e+00 - val_loss: 571.5676 - val_mae: 17.9564
Epoch 2/5
45/45 ━━━━━━━━━━━━━━━━━━━━ 24s 524ms/step - accuracy: 0.0014 - loss: 559.8862 - mae: 17.8113 - val_accuracy: 0.0028 - val_loss: 592.1206 - val_mae: 18.1766
Epoch 3/5
45/45 ━━━━━━━━━━━━━━━━━━━━ 24s 524ms/step - accuracy: 0.0021 - loss: 554.5771 - mae: 17.8372 - val_accuracy: 0.0000e+00 - val_loss: 581.4407 - val_mae: 18.6432
Epoch 4/5
30/45 ━━━━━━━━━━━━━━━━━━━━ 7s 492ms/step - accuracy: 0.0011 - loss: 517.6108 - mae: 17.2800